# Person 5 — Random Forest Classifier & Model Evaluation Pipeline

Pipeline Responsibility: Model Training & Held-Out Evaluation  
Model Assignment: Random Forest  

This notebook loads the full labelled dataset from `data/raw`, extracts features, fits Random Forest with bootstrap 95% CI evaluation, and writes results to `outputs/`.


In [ ]:
from pathlib import Path
import json, sys, time
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, confusion_matrix,
    classification_report, f1_score,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import joblib

MODEL_FOLDER = "random_forest"
HERE = Path.cwd().resolve()
ROOT = next(
    (
        p
        for p in (HERE, *HERE.parents)
        if (p / "data" / "raw").is_dir() and (p / "Basil_Leaf_ML_Workflow.ipynb").exists()
    ),
    None,
)
if ROOT is None:
    ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "raw").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find project root containing data/raw.")

MODEL_DIR = ROOT / "parts" / MODEL_FOLDER
OUTPUT_DIR = MODEL_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "parts"))
from _pipeline import CLASSES, FEATURE_VERSION, SEED, prepare_dataset  # noqa: E402

print("Python:", sys.executable)
print("Project root:", ROOT)
print("Model folder:", MODEL_DIR)
print("Outputs:", OUTPUT_DIR)

data = prepare_dataset(ROOT)
X_tr, y_tr = data["X_tr"], data["y_tr"]
X_te, y_te = data["X_te"], data["y_te"]
manifest = data["manifest"]
print(f"Unique images: {len(manifest)} | train: {len(X_tr)} | test: {len(X_te)}")
print("Class counts:", data["audit"]["class_counts"])
print("Dataset complete listed counts:", not data["audit"]["download_coverage"]["partial_dataset"])
print("Feature version:", FEATURE_VERSION)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

print("--- Person 5: Random Forest ---")
rf_model = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=2,
    max_features="sqrt",
    class_weight="balanced",
    random_state=SEED,
    n_jobs=2,
)
start_time = time.perf_counter()
rf_model.fit(X_tr, y_tr)
fit_time = time.perf_counter() - start_time
preds = rf_model.predict(X_te)
acc = accuracy_score(y_te, preds)
p, r, f1, _ = precision_recall_fscore_support(y_te, preds, average="macro", zero_division=0)
cm = confusion_matrix(y_te, preds, labels=CLASSES)

np.random.seed(SEED)
boot_f1s = []
n_samples = len(y_te)
for _ in range(1000):
    idx = np.random.choice(n_samples, size=n_samples, replace=True)
    boot_f1s.append(f1_score(y_te[idx], preds[idx], average="macro", zero_division=0))
ci_lower, ci_upper = np.percentile(boot_f1s, [2.5, 97.5])
print(f"Accuracy: {acc:.4f} | Macro F1: {f1:.4f}")
print(f"95% Bootstrap CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print("Confusion Matrix:\n", cm)

metrics = {
    "model_name": "Random Forest",
    "pipeline_stage": "Final Fit & Held-Out Evaluation",
    "n_estimators": 200,
    "accuracy": float(acc),
    "macro_f1": float(f1),
    "bootstrap_95_ci": [float(ci_lower), float(ci_upper)],
    "precision": float(p),
    "recall": float(r),
    "fit_time_seconds": float(fit_time),
    "confusion_matrix": cm.tolist(),
    "top_20_features": np.argsort(rf_model.feature_importances_)[-20:][::-1].tolist(),
    "n_train": int(len(X_tr)),
    "n_test": int(len(X_te)),
    "classes": CLASSES,
    "feature_version": FEATURE_VERSION,
}
(OUTPUT_DIR / "random_forest_metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
joblib.dump(rf_model, OUTPUT_DIR / "random_forest_model.joblib")
print("Saved outputs to:", OUTPUT_DIR)
